In [29]:
%matplotlib inline
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
from sklearn import svm, datasets
iris = datasets.load_iris()

In [3]:
df = pd.DataFrame(iris.data, columns= iris.feature_names)
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [4]:
df['flower'] = iris.target

In [7]:
df['flower'] = df['flower'].apply(lambda x: iris.target_names[x])

In [9]:
df['target'] = iris.target

In [10]:
df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower,target
0,5.1,3.5,1.4,0.2,setosa,0
1,4.9,3.0,1.4,0.2,setosa,0
2,4.7,3.2,1.3,0.2,setosa,0
3,4.6,3.1,1.5,0.2,setosa,0
4,5.0,3.6,1.4,0.2,setosa,0
...,...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica,2
146,6.3,2.5,5.0,1.9,virginica,2
147,6.5,3.0,5.2,2.0,virginica,2
148,6.2,3.4,5.4,2.3,virginica,2


### SVM

In [13]:
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3) #100 or 42
svm_model = SVC(kernel='rbf',  C=30, gamma='auto')
svm_model.fit(X_train, y_train)
svm_model.score(X_test, y_test)

0.9333333333333333

### Cross val score
Supplying models with different parameters to cross_val_score function with 5 fold cross validation

In [16]:
cross_val_score(SVC(kernel='linear', C=10, gamma='auto'), iris.data, iris.target, cv= 5)

array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

In [17]:
cross_val_score(SVC(kernel='rbf', C=10, gamma='auto'), iris.data, iris.target, cv= 5)

array([0.96666667, 1.        , 0.96666667, 0.96666667, 1.        ])

In [18]:
cross_val_score(SVC(kernel='rbf', C=20, gamma='auto'), iris.data, iris.target, cv= 5)

array([0.96666667, 1.        , 0.9       , 0.96666667, 1.        ])

In [21]:
kernels = ['rbf', 'linear']
C = [1, 10, 20]
avg_scores = {}
for ker_val in kernels: 
    for c_val in C: 
        cv_score = cross_val_score(SVC(kernel= ker_val, C= c_val, gamma= 'auto'), iris.data, iris.target)
        avg_scores[ker_val + '_' + str(c_val)] = np.average(cv_score)

avg_scores

{'rbf_1': np.float64(0.9800000000000001),
 'rbf_10': np.float64(0.9800000000000001),
 'rbf_20': np.float64(0.9666666666666668),
 'linear_1': np.float64(0.9800000000000001),
 'linear_10': np.float64(0.9733333333333334),
 'linear_20': np.float64(0.9666666666666666)}

### GridSearchCV

In [22]:
grid_s = GridSearchCV(SVC(gamma='auto'), {'C': [1, 10, 20], 
                                          'kernel': ['rbf', 'linear']}, cv= 5, return_train_score= False)
grid_s.fit(iris.data, iris.target)
grid_s.cv_results_

{'mean_fit_time': array([0.00141602, 0.00121593, 0.00116391, 0.00109491, 0.00115743,
        0.00105681]),
 'std_fit_time': array([2.21452920e-04, 1.41460556e-04, 6.78283049e-06, 4.58955703e-05,
        1.54060470e-05, 8.99770364e-06]),
 'mean_score_time': array([0.00106478, 0.00095186, 0.00090566, 0.00086141, 0.00090113,
        0.00083127]),
 'std_score_time': array([1.07197193e-04, 9.57070027e-05, 9.42956783e-06, 3.88272543e-05,
        6.47270217e-06, 4.32741215e-06]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value=np.str_('?'),
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
 

In [23]:
scores = pd.DataFrame(grid_s.cv_results_)
scores

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001416,0.000221,0.001065,0.000107,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
1,0.001216,0.000141,0.000952,0.000096,1,linear,"{'C': 1, 'kernel': 'linear'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
2,0.001164,0.000007,0.000906,0.000009,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,0.966667,1.0,0.980000,0.016330,1
3,0.001095,0.000046,0.000861,0.000039,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.966667,1.0,0.973333,0.038873,4
4,0.001157,0.000015,0.000901,0.000006,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.0,0.900000,0.966667,1.0,0.966667,0.036515,5
5,0.001057,0.000009,0.000831,0.000004,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.0,0.900000,0.933333,1.0,0.966667,0.042164,6


In [24]:
scores[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667


In [25]:
grid_s.best_params_

{'C': 1, 'kernel': 'rbf'}

In [26]:
grid_s.best_score_

np.float64(0.9800000000000001)

In [27]:
dir(grid_s)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__sklearn_tags__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_feature_names',
 '_check_n_features',
 '_check_refit_for_multimetric',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_routed_params_for_fit',
 '_get_scorers',
 '_get_tags',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr

##### Use RandomizedSearchCV to reduce number of iterations and with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. It helps reduce the cost of computation

In [30]:
rs_cv = RandomizedSearchCV(SVC(gamma= 'auto'), {'C': [1, 10, 20], 'kernel': ['rbf', 'linear']}, 
                          cv= 5, return_train_score= False, n_iter= 2)
rs_cv.fit(iris.data, iris.target)

RandomizedSearchCV(cv=5, estimator=SVC(gamma='auto'), n_iter=2,
                   param_distributions={'C': [1, 10, 20],
                                        'kernel': ['rbf', 'linear']})

In [33]:
results = pd.DataFrame(rs_cv.cv_results_)#[['param_C','param_kernel','mean_test_score']]
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kernel,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001405,0.000244,0.001054,0.000166,rbf,10,"{'kernel': 'rbf', 'C': 10}",0.966667,1.0,0.966667,0.966667,1.0,0.98,0.01633,1
1,0.001161,0.000055,0.000911,0.000058,linear,1,"{'kernel': 'linear', 'C': 1}",0.966667,1.0,0.966667,0.966667,1.0,0.98,0.01633,1


In [39]:
model_parameters = {'svm': {'model': SVC(gamma='auto'), 
                       'params': {'C': [1, 10, 20], 'kernel': ['rbf', 'linear']}},
               
               'random_forest': {'model': RandomForestClassifier(), 
                                'params': {'n_estimators': [1, 5, 50]}},
               
               'logistic_regression': {'model': LogisticRegression(solver='liblinear'), 
                                      'params': {'C': [1, 5, 10]}}
              }

In [40]:
score = []
for model_name, model_params in model_parameters.items(): 
    grid_search = GridSearchCV(model_params['model'], model_params['params'], cv=5 , return_train_score=False)
    grid_search.fit(iris.data, iris.target)
    score.append({'model': model_name, 
                  'best_score': grid_search.best_score_,
                  'best_params': grid_search.best_params_})

best_model = pd.DataFrame(score, columns=['model', 'best_score', 'best_params'])

In [41]:
best_model

,model,best_score,best_params
0,svm,0.980000,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.960000,{'n_estimators': 50}
2,logistic_regression,0.966667,{'C': 5}


##### Based on above, I can conclude that SVM with C=1 and kernel='rbf' is the best model for solving my problem of iris flower classification